In [2]:
import numpy as np

corr_matrix = np.array([
    [1.0, 0.8, 0.5, 0.8, 0.8, 0.4, 0.3, 0.8, 0.2, 0.3, 0.8, 0.4],
    [0.8, 1.0, 0.8, 0.8, 0.6, 0.8, 0.5, 0.5, 0.3, 0.5, 0.7, 0.6],
    [0.5, 0.8, 1.0, 0.8, 0.7, 0.6, 0.8, 0.7, 0.4, 0.7, 0.8, 0.7],
    [0.8, 0.8, 0.8, 1.0, 0.4, 0.3, 0.3, 0.8, 0.2, 0.3, 0.6, 0.5],
    [0.8, 0.6, 0.7, 0.4, 1.0, 0.5, 0.7, 0.6, 0.2, 0.8, 0.7, 0.5],
    [0.4, 0.8, 0.6, 0.3, 0.5, 1.0, 0.7, 0.7, 0.3, 0.8, 0.6, 0.6],
    [0.3, 0.5, 0.8, 0.3, 0.7, 0.7, 1.0, 0.6, 0.8, 0.6, 0.5, 0.6],
    [0.8, 0.5, 0.7, 0.8, 0.6, 0.7, 0.6, 1.0, 0.4, 0.8, 0.6, 0.8],
    [0.2, 0.3, 0.4, 0.2, 0.2, 0.3, 0.8, 0.4, 1.0, 0.5, 0.3, 0.6],
    [0.3, 0.5, 0.7, 0.3, 0.8, 0.8, 0.6, 0.8, 0.5, 1.0, 0.6, 0.7],
    [0.8, 0.7, 0.8, 0.6, 0.7, 0.6, 0.5, 0.6, 0.3, 0.6, 1.0, 0.7],
    [0.4, 0.6, 0.7, 0.5, 0.5, 0.6, 0.6, 0.8, 0.6, 0.7, 0.7, 1.0]
])

In [3]:
features = []
for i in range(corr_matrix.shape[0]):
    features.append(f"Var{i}")

In [4]:
threshold = 0.5  # Set your desired threshold

# Set values higher than threshold to 1, others to 0
binary_matrix = (corr_matrix > threshold).astype(int)

In [5]:
anchors = ["Var2", "Var7", "Var10"]
anchors_set = set(anchors)
anchors_indices = [features.index(feature) for feature in features if feature in anchors_set]
features_without_anchors_indices = [features.index(feature) for feature in features if feature not in anchors_set]

In [115]:
result = {
    str(idx): [
        i for i in np.where(binary_matrix[idx] == 1)[0] 
        if i != idx and i not in anchors_indices
    ]
    for idx in features_without_anchors_indices
}
print(result)


{'0': [1, 3, 4], '1': [0, 3, 4, 5, 11], '3': [0, 1], '4': [0, 1, 6, 9], '5': [1, 6, 9, 11], '6': [4, 5, 8, 9, 11], '8': [6, 11], '9': [4, 5, 6, 11], '11': [1, 5, 6, 8, 9]}


In [116]:
# Sort the dictionary by the length of the value lists (descending order)
sorted_result = dict(sorted(result.items(), key=lambda item: len(item[1]), reverse=False))
print(sorted_result)

{'3': [0, 1], '8': [6, 11], '0': [1, 3, 4], '4': [0, 1, 6, 9], '5': [1, 6, 9, 11], '9': [4, 5, 6, 11], '1': [0, 3, 4, 5, 11], '6': [4, 5, 8, 9, 11], '11': [1, 5, 6, 8, 9]}


In [117]:
#Group as value length
from collections import defaultdict
items = list(sorted_result.items())
length_groups = defaultdict(list)
for i, (k,v) in enumerate(items):
    length_groups[len(v)].append((k, v, i)) 
length_groups

defaultdict(list,
            {2: [('3', [0, 1], 0), ('8', [6, 11], 1)],
             3: [('0', [1, 3, 4], 2)],
             4: [('4', [0, 1, 6, 9], 3),
              ('5', [1, 6, 9, 11], 4),
              ('9', [4, 5, 6, 11], 5)],
             5: [('1', [0, 3, 4, 5, 11], 6),
              ('6', [4, 5, 8, 9, 11], 7),
              ('11', [1, 5, 6, 8, 9], 8)]})

In [118]:
advanced_results = []
prev_keys = set()

for length in sorted(length_groups.keys()):
    all_prev_keys = prev_keys.copy()
    print(f"length {length}, Prev_keys {all_prev_keys}")
    group = length_groups[length]

    def overlap_count(item):
        _, v, _ = item
        return sum(x in all_prev_keys for x in v)
    
    # Custom ordering algorithm
    def custom_sort(group):
        remaining = group[:]
        ordered = []
        used_keys = set()
    
        while remaining:
            # sort candidates by overlap_count (descending) then idx
            remaining.sort(key=lambda item: (-overlap_count(item), item[2]))
    
            # among ties, prefer those referencing already chosen keys
            best = None
            best_score = -1
            for candidate in remaining:
                _, v, _ = candidate
                score = sum(x in used_keys for x in v)
                if score > best_score:
                    best = candidate
                    best_score = score
            # place chosen item
            ordered.append(best)
            used_keys.add(int(best[0]))  # mark this var as "in front"
            remaining.remove(best)
    
        return ordered

    group_sorted = custom_sort(group)
    for k, v, idx in group_sorted:
        advanced_results.append((k, v))
        prev_keys.add(int(k))

length 2, Prev_keys set()
length 3, Prev_keys {8, 3}
length 4, Prev_keys {8, 0, 3}
length 5, Prev_keys {0, 3, 4, 5, 8, 9}


In [119]:
dict(advanced_results)

{'3': [0, 1],
 '8': [6, 11],
 '0': [1, 3, 4],
 '4': [0, 1, 6, 9],
 '9': [4, 5, 6, 11],
 '5': [1, 6, 9, 11],
 '1': [0, 3, 4, 5, 11],
 '11': [1, 5, 6, 8, 9],
 '6': [4, 5, 8, 9, 11]}

In [139]:
import numpy as np

def redesign_matrix_with_anchors(num_features, anchors, sorted_dict):
    """
    Redesign the binary matrix for Synthpop guidance with anchors as predictors for all features.
    Ensure strictly lower-triangular form (no predictors after diagonal).
    """
    # Final order = anchors first, then features in sorted_dict order
    feature_order = anchors + [int(k) for k in sorted_dict.keys()]
    order_map = {old: new for new, old in enumerate(feature_order)}
    n = len(feature_order)

    # Initialize matrix
    mat = np.zeros((n, n), dtype=int)

    # Fill anchors triangular structure
    for i in range(len(anchors)):
        for j in range(i):
            mat[i, j] = 1

    # Fill non-anchor features
    for f_str, correlated in sorted_dict.items():
        f = int(f_str)
        row = order_map[f]
        # 1) set all anchors as predictors (only if before this row)
        for a in range(len(anchors)):
            if a < row:
                mat[row, a] = 1
        # 2) set correlated features as predictors (only if before this row)
        for c in correlated:
            if c in order_map:
                col = order_map[c]
                if col < row:   # ensure lower-triangular
                    mat[row, col] = 1

    return mat, feature_order


# Example usage
anchors_indices = [2, 7, 10]
sorted_dict = {
 '3': [0, 1],
 '8': [6, 11],
 '0': [1, 3, 4],
 '4': [0, 1, 6, 9],
 '9': [4, 5, 6, 11],
 '5': [1, 6, 9, 11],
 '1': [0, 3, 4, 5, 11],
 '11': [1, 5, 6, 8, 9],
 '6': [4, 5, 8, 9, 11]
}

mat, order = redesign_matrix_with_anchors(12, anchors_indices, sorted_dict)

print("Feature order:", order)
print(mat)


Feature order: [2, 7, 10, 3, 8, 0, 4, 9, 5, 1, 11, 6]
[[0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 0]
 [1 1 0 0 0 0 0 0 0 0 0 0]
 [1 1 1 0 0 0 0 0 0 0 0 0]
 [1 1 1 0 0 0 0 0 0 0 0 0]
 [1 1 1 1 0 0 0 0 0 0 0 0]
 [1 1 1 0 0 1 0 0 0 0 0 0]
 [1 1 1 0 0 0 1 0 0 0 0 0]
 [1 1 1 0 0 0 0 1 0 0 0 0]
 [1 1 1 1 0 1 1 0 1 0 0 0]
 [1 1 1 0 1 0 0 1 1 1 0 0]
 [1 1 1 0 1 0 1 1 1 0 1 0]]


In [140]:
for i in range(mat.shape[0]):
    print(np.sum(mat[i,:]))

0
1
2
3
3
4
4
4
4
7
7
8


In [131]:
features_indices = [features.index(feature) for feature in features]
list(zip(features, features_indices))

[('Var0', 0),
 ('Var1', 1),
 ('Var2', 2),
 ('Var3', 3),
 ('Var4', 4),
 ('Var5', 5),
 ('Var6', 6),
 ('Var7', 7),
 ('Var8', 8),
 ('Var9', 9),
 ('Var10', 10),
 ('Var11', 11)]

In [136]:
import pandas as pd

# Example: you provide this zip
# var_index_zip = [('Var0', 0), ('Var1', 1), ...]
features_indices = [features.index(feature) for feature in features]
var_index_zip = list(zip(features, features_indices))


# Create a dict: index -> variable name
index_to_var = {idx: name for name, idx in var_index_zip}

# Reorder variable names according to feature_order
ordered_var_names = [index_to_var[idx] for idx in order]

# Convert np.array to DataFrame
df = pd.DataFrame(mat, index=ordered_var_names, columns=ordered_var_names)
df


,Var2,Var7,Var10,Var3,Var8,Var0,Var4,Var9,Var5,Var1,Var11,Var6
Var2,0,0,0,0,0,0,0,0,0,0,0,0
Var7,1,0,0,0,0,0,0,0,0,0,0,0
Var10,1,1,0,0,0,0,0,0,0,0,0,0
Var3,1,1,1,0,0,1,0,0,0,1,0,0
Var8,1,1,1,0,0,0,0,0,0,0,1,1
Var0,1,1,1,1,0,0,1,0,0,1,0,0
Var4,1,1,1,0,0,1,0,1,0,1,0,1
Var9,1,1,1,0,0,0,1,0,1,0,1,1
Var5,1,1,1,0,0,0,0,1,0,1,1,1
Var1,1,1,1,1,0,1,1,0,1,0,1,0
